# 案例 02 · 内存泄漏触发 OOM Kill，服务反复重启

> 沙箱实验：真实进程 + 阿里云 OpenAPI 模拟（响应带 `Mock: true`）。
> 云产品字段/命令与生产一致，资源规格与账单为模拟值。

## 场景

笔记详情缓存改版后引入对象泄漏，进程 RSS 持续上涨，超过 220MB 后被内核 OOM Kill，systemd 自动拉起，形成“重启—再涨—再杀”的循环，期间接口间歇性 502。

## 练习目标

- 用 `dmesg` 与重启次数确认 OOM Kill
- 摘流量后重启止血，保留现场
- 用 systemd `MemoryMax` 限制内存并配置告警
- 用 `memory_guard.py` 完成自动化处置

## 前置

- 仓库根目录执行 `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
- 使用该 venv 的 Python kernel 打开本 Notebook；或运行 `tools/run_all_labs.py` 批量执行


In [ ]:
# 环境准备：定位仓库根目录并加载沙箱
import json, os, signal, ssl, socket, sys, time, gzip, shutil, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "labs" / "common" / "xhs_lab.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("请在 xhs-backend-cloud-ops-lab 仓库内运行本 Notebook")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from labs.common import Lab

lab = Lab("case-02-memory-oom")
lab.reset(start_api=True)

lab.summary()


## 步骤 1 · 异常重现

下面注入一个可复现的真实故障，观察它对业务的影响。

In [ ]:
lab.inject_memory_leak(limit_mb=220)
print("OOM 次数:", lab.state["faults"]["memory"]["oom"])
print("服务重启次数:", lab.state["services"]["xhs-note-api"]["restarts"])


## 步骤 2 · 取证

按“入口 → 主机 → 进程 → 依赖 → 日志”的分层顺序收集证据，不要一上来就重启。

**内核 OOM 记录**

In [ ]:
print(lab.sh("cat /var/log/xhs/dmesg.log"))

**服务状态**

In [ ]:
print(lab.sh("systemctl status xhs-note-api"))

**运行指标**

In [ ]:
print(json.dumps(lab.api_stats(), ensure_ascii=False, indent=2))

**应用日志**

In [ ]:
print(lab.sh("tail -n 5 /var/log/xhs/note-api.log"))

## 步骤 3 · 判断

把证据收敛成一句结论，决定止血动作。

In [ ]:
st = lab.api_stats()
fault = lab.state["faults"]["memory"]
print("当前 RSS:", st.get("rss_mb"), "MB；水位线:", fault["limit_mb"], "MB；OOM:", fault["oom"], "次")
print("重启次数:", lab.state["services"]["xhs-note-api"]["restarts"])
print("判断：RSS 单调上涨且触发 OOM → 内存泄漏，而非流量突增")


## 步骤 4 · 手工处置（先止血）

执行最小必要的变更，优先恢复业务，再考虑根治。

In [ ]:
svc = lab.state["services"]["xhs-note-api"]
lb = lab.state["slb"]["LoadBalancerId"]

# 1) 先摘流量，避免重启期间请求打到半启动实例
lab.aliyun(f"slb RemoveBackendServers --LoadBalancerId {lb} --BackendServers i-bp1xhsapi01")
# 2) 重启止血（保留 dmesg / 日志现场）
lab.kill_service()
lab.start_api(port=svc["port"], nofile=256)
# 3) 加内存上限，防止单进程拖垮整机
lab.write("/etc/systemd/system/xhs-note-api.service.d/memory.conf",
          "[Service]\nMemoryMax=1G\nMemoryHigh=800M\nRestart=always\nRestartSec=2\n")
# 4) 健康后挂回
lab.aliyun(f"slb AddBackendServers --LoadBalancerId {lb} --BackendServers i-bp1xhsapi01")
time.sleep(1)
print("重启后 RSS:", lab.api_stats().get("rss_mb"), "MB")


## 步骤 5 · 验证

用可执行的断言确认恢复，而不是“看起来好了”。

In [ ]:
lab.check('RSS 已回落', bool(lab.api_stats().get('rss_mb', 999) < 120))
lab.check('发生过 OOM Kill', bool(lab.state['services']['xhs-note-api']['restarts'] >= 1))
lab.check('服务健康', bool('HTTP_CODE=200' in lab.curl(lab.api + '/healthz')))

## 步骤 6 · 自动化沉淀

脚本读取 RSS 与 OOM 记录，先经 SLB 摘除后端，再重启服务并写入 `MemoryMax`，健康后自动挂回，最后校验 RSS 与探活。

下面重置沙箱、重新注入故障，然后直接调用生产可用的自动化脚本（`--apply`；缺省为 dry-run）。

In [ ]:
lab.reset(start_api=True)
lab.inject_memory_leak(limit_mb=220)
out = lab.auto('memory_guard.py', '--apply')
print(out)

In [ ]:
lab.check('自动化处置返回 REMEDIATED', 'RESULT: REMEDIATED' in out)
print('\n自动化脚本已可作为 cron / 事件总线触发器复用')

## 复盘与思考题

1. `dmesg` 中哪些字段能证明是 OOM Kill 而不是应用主动退出？
2. 为什么先摘流量再重启？
3. MemoryMax 与容器 memory limit 同时存在时以哪个为准？

## 收尾

沙箱资源会在本 Cell 中停止；重新运行请从第 1 个代码 Cell 开始。


In [ ]:
lab.stop_all()
print('沙箱已停止')